## Setup

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "transfermarkt"

def gold(t):
    return spark.read.parquet(f"/Volumes/{CATALOG}/{SCHEMA}/gold/{t}")

fact_appearance = gold("fact_appearance")
dim_player = gold("dim_player")

## Aggregate to one row per player

In [0]:
player_agg = (
    fact_appearance
    .groupBy("player_key")
    .agg(
        F.sum("goals").alias("total_goals"),
        F.sum("assists").alias("total_assists"),
        F.sum("yellow_cards").alias("total_yellow"),
        F.sum("red_cards").alias("total_red"),
        F.sum("minutes_played").alias("total_minutes"),
        F.count("*").alias("total_appearances"),
    )
)
print("Players with at least one appearance:", player_agg.count())

## The actual distribution of appearances — deciding the minimum-sample threshold from this, not a guess

In [0]:
player_agg.select("total_appearances").summary().show()
player_agg.groupBy((F.col("total_appearances") / 10).cast("int").alias("appearances_bucket")) \
    .count().orderBy("appearances_bucket").show(20)

## Minutes, not appearances, as the real sample-size floor
Total minutes is the actual exposure variable behind a per-90 rate — two players with the same appearance count can have very different amounts of real playing time behind that number. 900 minutes (10 full matches) is the standard analytics convention for a minimally stable rate; testing a few candidates against the real distribution rather than assuming that number transfers directly to a career-total (not single-season) context.

In [0]:
player_agg.select("total_minutes").summary().show()

for threshold in [450, 900, 1800, 3600]:
    kept = player_agg.filter(F.col("total_minutes") >= threshold).count()
    print(f"total_minutes >= {threshold} ({threshold/90:.0f} full matches): "
          f"{kept} players kept ({kept/player_agg.count()*100:.1f}%)")

## Filter to the reliable population, compute per-90 rates
900 minutes (10 full matches), justified above. minutes_per_appearance carries over the same "role size" signal used in regression — a squad-rotation player and a guaranteed starter can have the same total minutes while looking very different game-to-game.

In [0]:
clustering_base = player_agg.filter(F.col("total_minutes") >= 900)
print("Players retained (>=900 total minutes):", clustering_base.count())

clustering_base = (
    clustering_base
    .withColumn("goals_per90", F.col("total_goals") / F.col("total_minutes") * 90)
    .withColumn("assists_per90", F.col("total_assists") / F.col("total_minutes") * 90)
    .withColumn("yellow_per90", F.col("total_yellow") / F.col("total_minutes") * 90)
    .withColumn("red_per90", F.col("total_red") / F.col("total_minutes") * 90)
    .withColumn("minutes_per_appearance", F.col("total_minutes") / F.col("total_appearances"))
)

## Check the rate distributions before trusting them as clustering inputs
Red cards specifically are rare events over a career — worth checking whether red_per90 has enough real variation to mean anything in a distance-based method like K-Means, or whether it's effectively a constant near-zero for almost everyone, in which case it wouldn't be separating players so much as adding noise to the distance calculation.

In [0]:
clustering_base.select("goals_per90", "assists_per90", "yellow_per90", "red_per90", "minutes_per_appearance").summary().show()

print("Players with zero career red cards:", clustering_base.filter(F.col("total_red") == 0).count(),
      "out of", clustering_base.count())

## Final feature set: drop red_per90, scale, and write
Four features survive: goals_per90, assists_per90, yellow_per90, minutes_per_appearance — scoring output, creative output, discipline, and role size. StandardScaler matters more here than it did for regression's regularization reason — K-Means uses raw Euclidean distance directly, so without scaling, minutes_per_appearance (spanning ~20-107) would swamp the goals/assists/yellow rates (spanning roughly 0-1) in every distance calculation, regardless of which actually separates players better.

In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

feature_cols = ["goals_per90", "assists_per90", "yellow_per90", "minutes_per_appearance"]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True)

assembled = assembler.transform(clustering_base)
scaler_model = scaler.fit(assembled)
clustering_final = scaler_model.transform(assembled)

clustering_final.write.mode("overwrite").parquet(f"/Volumes/{CATALOG}/{SCHEMA}/ml/clustering_features_final")
print("Final row count:", clustering_final.count())
clustering_final.select("player_key", *feature_cols).show(10)